# P1A development window: the liquidity test's calibration (draft test, not registered)

This notebook reads **only the development window, 1 January to 30 June 2024**. The code refuses any later date. It
measures what the power simulation needs (step 2 of section 7 in `P1A_liquidity_spec_DRAFT.md`):
- how many pools and token pairs pass the activity floors;
- how reliable the weekly response estimate is;
- how strongly the instrument moves composition;
- how noisy depth is.

On purpose, it computes **no estimate of composition's effect on depth**, so the test's outcome is not previewed.
Please don't compute it from the saved files either.

**Before you start.** Create a Google Cloud project at console.cloud.google.com and copy its project ID. The free
BigQuery sandbox is enough; no billing account is needed.

**Cost.** Step 3 only estimates how much data each query would scan, which costs nothing. Step 4 refuses to run unless
the total is below 950 GB (the sandbox gives 1,000 GB free each month) and you have set `I_ACCEPT_THE_SCAN = True`.
Each query is also capped, so BigQuery stops it rather than bill more.

**Time.** About one to two hours. Everything is saved to `MyDrive/P1A_dev`. If the session disconnects, run the cells
again from the top; finished steps are skipped.

**At the end**, copy the JSON printed by step 7 into the chat with Claude.


In [ ]:
# Step 1. Settings
PROJECT_ID = ""                          # your Google Cloud project ID
OUT_DIR = "/content/drive/MyDrive/P1A_dev"
I_ACCEPT_THE_SCAN = False                # set to True after reading the estimate in step 3, then rerun

In [ ]:
# Step 2. Setup: Drive, Google sign-in, and the pipeline code (checked by its SHA-256)
!pip -q install pycryptodome
import hashlib, importlib, json, os, sys
from google.colab import auth, drive
drive.mount("/content/drive")
auth.authenticate_user()
P1A_DEV_SHA256 = "108af4d2f6aa612f8b60f28f88b13fbfdff7bab3124e7397edb3c242ab1f87d4"
P1A_DEV_SRC = "\"\"\"P1A development window: extraction and calibration for the draft liquidity test (not registered).\n\nRuns in Google Colab under the first author's Google account; the free BigQuery sandbox is enough. It reads only the\ndevelopment window, 2024-01-01 to 2024-06-30. Every query and every download is checked against that window, and\nanything later is refused. It computes what the power simulation needs (step 2 of section 7 in\nP1A_liquidity_spec_DRAFT.md). On purpose, it computes no estimate of composition's effect on depth, so the test's\noutcome is not previewed.\n\nStages. Each writes to OUT and is skipped when its output exists, so a disconnected session resumes where it stopped.\n  candidate_pools()       Uniswap v3 pool addresses for the candidate pairs, by CREATE2 (no query)\n  estimate(client, ...)   dry runs: the bytes each query would scan (no cost)\n  extract(client, ...)    block timestamps, then the Swap/Mint/Burn logs of the candidate pools with each transaction's\n                          sender, month by month; each query is capped by maximum_bytes_billed; decoded at once\n  needed_symbols(...)     the Binance symbols of the pools that pass the swap-count floor\n  binance(...)            one-second klines of those symbols from data.binance.vision\n  build_panel(...)        pool-week measures: response halves R1 and R2, composition C, depth D, controls, instrument Z\n  calibrate(...)          the power simulation's inputs, written to calibration.json\n\"\"\"\nimport hashlib\nimport io\nimport json\nimport math\nimport time\nimport urllib.error\nimport urllib.request\nimport zipfile\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\n# ----------------------------------------------------------------------------------------------------------------------\n# The window. 2024-01-01 is a Monday, so the window is 26 whole weeks.\nDEV_START, DEV_END = \"2024-01-01\", \"2024-07-01\"          # end exclusive\nT0, T1 = 1704067200, 1719792000                           # the same instants in Unix seconds\nNSEC, WEEK, WEEKS = T1 - T0, 7 * 86400, 26\nMONTHS = [(\"2024-01-01\", \"2024-02-01\"), (\"2024-02-01\", \"2024-03-01\"), (\"2024-03-01\", \"2024-04-01\"),\n          (\"2024-04-01\", \"2024-05-01\"), (\"2024-05-01\", \"2024-06-01\"), (\"2024-06-01\", \"2024-07-01\")]\nEXPOSURE_WEEKS = 13            # instrument exposures from weeks 1-13; its first stage is measured on weeks 14-26\n\n# Provisional floors for the exploration. The registered values are fixed after it.\nMIN_SWAPS_WEEK = 100           # swaps in every one of the 26 weeks\nMIN_ACTIVE_SHARE = 0.90        # share of seconds in which each Binance leg trades (the spec's floor)\nMAX_MEDIAN_BASIS_BPS = 50.0    # data check: a token or decimals mismatch shows up as a huge constant basis\nVOLUME_FLOORS_USD = (1e6, 5e6, 20e6)   # median weekly volume; counts reported at each, the first one used\nWINSOR = (1.0, 99.0)           # the basis is winsorized at these percentiles within each pool-week-half\nN_BOTS = 50                    # senders used in the shift-share instrument\nDEPTH_BAND = 0.01              # depth within +-1% of the pool price\n\n# Uniswap v3 on Ethereum mainnet\nFACTORY = \"1f98431c8ad98523631ae4a59f267346ea31f984\"\nINIT_CODE_HASH = \"e34f199b19b2b4f47f68442619d555527d244f78a3297ea89325f843f87b8b54\"\nFEES = (100, 500, 3000, 10000)\nTOPIC_SWAP = \"0xc42079f94a6350d7e6235f29174924f928cc2ac818eb64fed8004e115fbcca67\"  # Swap(address,address,int256,int256,uint160,uint128,int24)\nTOPIC_MINT = \"0x7a53080ba414158be7ec69b987b5fb7d07dee101fe85488f0853ae16239d0bde\"  # Mint(address,address,int24,int24,uint128,uint256,uint256)\nTOPIC_BURN = \"0x0c396cd989a39f4459b5fa1aed6a9a8dcdbc45908acfd67e028cd568da98982c\"  # Burn(address,int24,int24,uint128,uint256,uint256)\nKNOWN_POOLS = {(\"USDC\", \"WETH\", 500): \"0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640\",\n               (\"USDC\", \"WETH\", 3000): \"0x8ad599c3a0ff1de082011efddc58f1908eb6e6d8\",\n               (\"WBTC\", \"WETH\", 3000): \"0xcbcdf9626bc03e24f779434178a73a0b4bad62ed\",\n               (\"WETH\", \"USDT\", 500): \"0x11b815efb8f581194ae79006d24e0d814b7697f6\",\n               (\"WBTC\", \"WETH\", 500): \"0x4585fe77225b41b697c938b018e2ac67ac5a20c0\"}\nDATASET = \"bigquery-public-data.crypto_ethereum\"\nBUDGET_BYTES = 950 * 10 ** 9   # below the sandbox's free 1 TB a month\nBINANCE_URL = \"https://data.binance.vision/data/spot/monthly/klines/{s}/1s/{s}-1s-{m}.zip\"\nBINANCE_DAILY_URL = \"https://data.binance.vision/data/spot/daily/klines/{s}/1s/{s}-1s-{d}.zip\"\n\n# Candidate tokens: symbol -> (mainnet address, decimals, Binance spot symbol; None means USDT itself, price 1).\n# Tokens that migrated or merged during 2024-2026 (MATIC, RNDR, OCEAN, AGIX) are left out so the pool list can be\n# reused on the test window. A wrong address only yields pools without events; a wrong decimals value is caught by\n# the median-basis check.\nQUOTES = (\"WETH\", \"USDC\", \"USDT\", \"WBTC\")\nTOKENS = {\n    \"WETH\": (\"0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2\", 18, \"ETHUSDT\"),\n    \"USDC\": (\"0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48\", 6, \"USDCUSDT\"),\n    \"USDT\": (\"0xdac17f958d2ee523a2206206994597c13d831ec7\", 6, None),\n    \"WBTC\": (\"0x2260fac5e5542a773aa44fbcfedf7c193bc2c599\", 8, \"BTCUSDT\"),\n    \"LINK\": (\"0x514910771af9ca656af840dff83e8264ecf986ca\", 18, \"LINKUSDT\"),\n    \"UNI\": (\"0x1f9840a85d5af5bf1d1762f925bdaddc4201f984\", 18, \"UNIUSDT\"),\n    \"AAVE\": (\"0x7fc66500c84a76ad7e9c93437bfc5ac33e2ddae9\", 18, \"AAVEUSDT\"),\n    \"MKR\": (\"0x9f8f72aa9304c8b593d555f12ef6589cc3a579a2\", 18, \"MKRUSDT\"),\n    \"LDO\": (\"0x5a98fcbea516cf06857215779fd812ca3bef1b32\", 18, \"LDOUSDT\"),\n    \"PEPE\": (\"0x6982508145454ce325ddbe47a25d4ec3d2311933\", 18, \"PEPEUSDT\"),\n    \"SHIB\": (\"0x95ad61b0a150d79219dcf64e1e6cc01f0b64c4ce\", 18, \"SHIBUSDT\"),\n    \"CRV\": (\"0xd533a949740bb3306d119cc777fa900ba034cd52\", 18, \"CRVUSDT\"),\n    \"SNX\": (\"0xc011a73ee8576fb46f5e1c5751ca3b9fe0af2a6f\", 18, \"SNXUSDT\"),\n    \"COMP\": (\"0xc00e94cb662c3520282e6f5717214004a7f26888\", 18, \"COMPUSDT\"),\n    \"ENS\": (\"0xc18360217d8f7ab5e7c516566761ea12ce7f9d72\", 18, \"ENSUSDT\"),\n    \"GRT\": (\"0xc944e90c64b2c07662a292be6244bdf05cda44a7\", 18, \"GRTUSDT\"),\n    \"1INCH\": (\"0x111111111117dc0aa78b770fa6a738034120c302\", 18, \"1INCHUSDT\"),\n    \"APE\": (\"0x4d224452801aced8b2f0aebe155379bb5d594381\", 18, \"APEUSDT\"),\n    \"SUSHI\": (\"0x6b3595068778dd592e39a122f4f5a5cf09c90fe2\", 18, \"SUSHIUSDT\"),\n    \"FET\": (\"0xaea46a60368a7bd060eec7df8cba43b7ef41ad85\", 18, \"FETUSDT\"),\n    \"IMX\": (\"0xf57e7e7c23978c3caec3c3548e3d615c346e79ff\", 18, \"IMXUSDT\"),\n    \"BLUR\": (\"0x5283d291dbcf85356a21ba090e6db59121208b44\", 18, \"BLURUSDT\"),\n    \"WLD\": (\"0x163f8c2467924be0ae7b5347228cabf260318753\", 18, \"WLDUSDT\"),\n    \"ARB\": (\"0xb50721bcf8d664c30412cfbc6cf7a15145234ad1\", 18, \"ARBUSDT\"),\n    \"ENA\": (\"0x57e114b691db790c35207b2e685d4a43181e6061\", 18, \"ENAUSDT\"),\n    \"ETHFI\": (\"0xfe0c30065b384f05761f15d0cc899d4f9f9cc0eb\", 18, \"ETHFIUSDT\"),\n    \"STRK\": (\"0xca14007eff0db1f8135f4c25b34de49ab0d42766\", 18, \"STRKUSDT\"),\n    \"PENDLE\": (\"0x808507121b80c02388fad14726482e061b8da827\", 18, \"PENDLEUSDT\"),\n    \"FLOKI\": (\"0xcf0c122c6b73ff809c693db761e7baebe62b6a2e\", 9, \"FLOKIUSDT\"),\n    \"INJ\": (\"0xe28b3b32b6c345a34ff64674606124dd5aceca30\", 18, \"INJUSDT\"),\n    \"MANA\": (\"0x0f5d2fb29fb7d3cfee444a200298f468908cc942\", 18, \"MANAUSDT\"),\n    \"SAND\": (\"0x3845badade8e6dff049820680d1f14bd3903a5d0\", 18, \"SANDUSDT\"),\n    \"AXS\": (\"0xbb0e17ef65f82ab018d8edd776e8dd940327b28b\", 18, \"AXSUSDT\"),\n    \"CHZ\": (\"0x3506424f91fd33084466f402d5d97f05f8e3b4af\", 18, \"CHZUSDT\"),\n    \"LRC\": (\"0xbbbbca6a901c926f240b89eacb641d8aec7aeafd\", 18, \"LRCUSDT\"),\n    \"BAT\": (\"0x0d8775f648430679a709e98d2b0cb6250d2887ef\", 18, \"BATUSDT\"),\n    \"YFI\": (\"0x0bc529c00c6401aef6d220be8c6ea1667f6ad93e\", 18, \"YFIUSDT\"),\n    \"SSV\": (\"0x9d65ff81a3c488d585bbfb0bfe3c7707c7917f54\", 18, \"SSVUSDT\"),\n    \"ZRX\": (\"0xe41d2489571d322189246dafa5ebde1f4699f498\", 18, \"ZRXUSDT\"),\n    \"BAL\": (\"0xba100000625a3754423978a60c9317c58a424e3d\", 18, \"BALUSDT\"),\n    \"KNC\": (\"0xdefa4e8a7bcba345f687a2f1456f5edd9ce97202\", 18, \"KNCUSDT\"),\n    \"STG\": (\"0xaf5191b0de278c7286d6c7cc6ab6bb8a73ba2cd6\", 18, \"STGUSDT\"),\n    \"GALA\": (\"0xd1d2eb1b1e90b638588728b4130137d262c87cae\", 8, \"GALAUSDT\"),\n    \"MASK\": (\"0x69af81e73a73b40adf4f3d4223cd9b1ece623074\", 18, \"MASKUSDT\"),\n    \"ILV\": (\"0x767fe9edc9e0df98e07454847909b5e959d7ca0e\", 18, \"ILVUSDT\"),\n    \"RSR\": (\"0x320623b8e4ff03373931769a31fc52a4e78b5d70\", 18, \"RSRUSDT\"),\n    \"CVX\": (\"0x4e3fbd56cd56c3e72c1403e103b45db9da5b9d2b\", 18, \"CVXUSDT\"),\n    \"LPT\": (\"0x58b6a8a3302369daec383334672404ee733ab239\", 18, \"LPTUSDT\"),\n    \"API3\": (\"0x0b38210ea11411557c13457d4da7dc6ea731b88a\", 18, \"API3USDT\"),\n    \"UMA\": (\"0x04fa0d235c4abf4bcf4787af4cf447de572ef828\", 18, \"UMAUSDT\"),\n    \"PAXG\": (\"0x45804880de22913dafe09f4980848ece6ecbaf78\", 18, \"PAXGUSDT\"),\n    \"WOO\": (\"0x4691937a7508860f876c9c0a2a617e7d9e945d4b\", 18, \"WOOUSDT\"),\n    \"ANKR\": (\"0x8290333cef9e6d528dd5618fb97a76f268f3edd4\", 18, \"ANKRUSDT\"),\n    \"ENJ\": (\"0xf629cbd94d3791c9250152bd8dfbdf380e2a3b9c\", 18, \"ENJUSDT\"),\n    \"MEME\": (\"0xb131f4a55907b10d1f0a50d8ab8fa09ec342cd74\", 18, \"MEMEUSDT\"),\n}\n\n\ndef in_dev(a, b):\n    \"\"\"Refuse any interval outside the development window.\"\"\"\n    if not (DEV_START <= a < b <= DEV_END):\n        raise ValueError(f\"refused: {a} to {b} is outside the development window {DEV_START} to {DEV_END}\")\n\n\n# ----------------------------------------------------------------------------------------------------------------------\n# Pools, by CREATE2\n\ndef _keccak(b):\n    from Crypto.Hash import keccak  # pycryptodome\n    h = keccak.new(digest_bits=256)\n    h.update(b)\n    return h.digest()\n\n\ndef pool_address(addr_a, addr_b, fee):\n    a, b = sorted([addr_a.lower()[2:], addr_b.lower()[2:]])\n    salt = _keccak(bytes(12) + bytes.fromhex(a) + bytes(12) + bytes.fromhex(b) + int(fee).to_bytes(32, \"big\"))\n    return \"0x\" + _keccak(b\"\\xff\" + bytes.fromhex(FACTORY) + salt + bytes.fromhex(INIT_CODE_HASH))[12:].hex()\n\n\ndef candidate_pools():\n    \"\"\"Every candidate token against each quote token, and the quote tokens against each other, at every fee tier.\"\"\"\n    pairs = set()\n    for s in TOKENS:\n        for q in QUOTES:\n            if s != q:\n                pairs.add(tuple(sorted((s, q))))\n    rows = []\n    for s, q in sorted(pairs):\n        (aa, da, ba), (ab, db, bb) = TOKENS[s], TOKENS[q]\n        (s0, a0, d0, b0), (s1, a1, d1, b1) = sorted([(s, aa, da, ba), (q, ab, db, bb)], key=lambda t: t[1])\n        for fee in FEES:\n            rows.append({\"pool\": pool_address(a0, a1, fee), \"sym0\": s0, \"sym1\": s1, \"fee\": fee, \"dec0\": d0,\n                         \"dec1\": d1, \"bin0\": b0, \"bin1\": b1, \"pair\": f\"{s0}/{s1}\"})\n    df = pd.DataFrame(rows)\n    for (x, y, fee), addr in KNOWN_POOLS.items():   # self-test of the CREATE2 computation\n        got = df[(df.fee == fee) & (df.pair.isin([f\"{x}/{y}\", f\"{y}/{x}\"]))].pool.tolist()\n        assert got == [addr], (x, y, fee, got)\n    return df\n\n\n# ----------------------------------------------------------------------------------------------------------------------\n# BigQuery\n\ndef sql_blocks(a, b):\n    in_dev(a, b)\n    return (f\"SELECT number, UNIX_SECONDS(timestamp) AS ts, base_fee_per_gas\\n\"\n            f\"FROM `{DATASET}.blocks`\\n\"\n            f\"WHERE timestamp >= TIMESTAMP '{a}' AND timestamp < TIMESTAMP '{b}'\")\n\n\ndef sql_logs(a, b, pools):\n    in_dev(a, b)\n    plist = \", \".join(f\"'{p}'\" for p in pools)\n    swap = f\"l.topics[SAFE_OFFSET(0)] = '{TOPIC_SWAP}'\"\n    return (f\"SELECT l.block_number, l.log_index, l.address, l.topics[SAFE_OFFSET(0)] AS t0,\\n\"\n            f\"  IF({swap}, NULL, l.topics[SAFE_OFFSET(2)]) AS t2, IF({swap}, NULL, l.topics[SAFE_OFFSET(3)]) AS t3,\\n\"\n            f\"  l.data, IF({swap}, t.from_address, NULL) AS from_address\\n\"\n            f\"FROM `{DATASET}.logs` AS l\\n\"\n            f\"JOIN `{DATASET}.transactions` AS t ON t.hash = l.transaction_hash\\n\"\n            f\"WHERE l.block_timestamp >= TIMESTAMP '{a}' AND l.block_timestamp < TIMESTAMP '{b}'\\n\"\n            f\"  AND t.block_timestamp >= TIMESTAMP '{a}' AND t.block_timestamp < TIMESTAMP '{b}'\\n\"\n            f\"  AND l.address IN ({plist})\\n\"\n            f\"  AND l.topics[SAFE_OFFSET(0)] IN ('{TOPIC_SWAP}', '{TOPIC_MINT}', '{TOPIC_BURN}')\")\n\n\ndef sql_tokens():\n    alist = \", \".join(f\"'{a}'\" for a, _, _ in TOKENS.values())\n    return f\"SELECT address, symbol, decimals FROM `{DATASET}.tokens` WHERE address IN ({alist})\"\n\n\ndef queries(pools):\n    q = [(\"tokens\", sql_tokens()), (\"blocks\", sql_blocks(DEV_START, DEV_END))]\n    q += [(f\"logs {a[:7]}\", sql_logs(a, b, pools)) for a, b in MONTHS]\n    return q\n\n\ndef estimate(client, pools):\n    \"\"\"Dry runs. Returns [(name, bytes)]; costs nothing.\"\"\"\n    from google.cloud import bigquery\n    out = []\n    for name, sql in queries(pools):\n        job = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))\n        out.append((name, int(job.total_bytes_processed)))\n    return out\n\n\ndef _run(client, sql, est_bytes):\n    from google.cloud import bigquery\n    cap = int(est_bytes * 1.05) + 10 ** 9           # fails without charge if BigQuery would bill more\n    job = client.query(sql, job_config=bigquery.QueryJobConfig(maximum_bytes_billed=cap))\n    try:\n        return job.result().to_dataframe(create_bqstorage_client=True)\n    except Exception:                               # the Storage Read API may be off in a sandbox project\n        return job.result().to_dataframe(create_bqstorage_client=False)\n\n\ndef _word(h, i):\n    return int(h[2 + 64 * i: 66 + 64 * i], 16)\n\n\ndef _signed(v):\n    return v - (1 << 256) if v >= (1 << 255) else v\n\n\ndef decode(df):\n    \"\"\"Raw logs -> one compact row per event. kind: 0 swap, 1 mint, 2 burn.\"\"\"\n    n = len(df)\n    kind = np.empty(n, np.int8)\n    a0 = np.zeros(n)\n    a1 = np.zeros(n)\n    sp = np.full(n, np.nan)\n    liq = np.full(n, np.nan)\n    amt = np.zeros(n)\n    tick = np.zeros(n, np.int64)\n    tl = np.zeros(n, np.int64)\n    tu = np.zeros(n, np.int64)\n    for j, (t0, t2, t3, data) in enumerate(zip(df[\"t0\"].tolist(), df[\"t2\"].tolist(), df[\"t3\"].tolist(),\n                                               df[\"data\"].tolist())):\n        if t0 == TOPIC_SWAP:\n            kind[j] = 0\n            a0[j], a1[j] = float(_signed(_word(data, 0))), float(_signed(_word(data, 1)))\n            sp[j], liq[j] = float(_word(data, 2)), float(_word(data, 3))\n            tick[j] = _signed(_word(data, 4))\n        else:\n            kind[j] = 1 if t0 == TOPIC_MINT else 2\n            tl[j], tu[j] = _signed(int(t2, 16)), _signed(int(t3, 16))\n            amt[j] = float(_word(data, 1) if t0 == TOPIC_MINT else _word(data, 0))\n    return pd.DataFrame({\"block_number\": df[\"block_number\"].to_numpy(np.int64), \"log_index\": df[\"log_index\"].to_numpy(np.int64),\n                         \"pool\": df[\"address\"].str.lower().astype(\"category\"), \"kind\": kind,\n                         \"sender\": df[\"from_address\"].fillna(\"\").str.lower().astype(\"category\"),\n                         \"a0\": a0, \"a1\": a1, \"sqrtp\": sp, \"liq\": liq, \"amount\": amt, \"tick\": tick,\n                         \"tick_lower\": tl, \"tick_upper\": tu})\n\n\ndef extract(client, out, pools, estimates, log=print):\n    \"\"\"Runs the queries whose outputs are missing; each is capped at its dry-run estimate.\"\"\"\n    out = Path(out)\n    (out / \"events\").mkdir(parents=True, exist_ok=True)\n    est = dict(estimates)\n    q = dict(queries(pools))\n    if not (out / \"tokens.csv\").exists():\n        _run(client, q[\"tokens\"], est[\"tokens\"]).to_csv(out / \"tokens.csv\", index=False)\n    if not (out / \"blocks.parquet\").exists():\n        b = _run(client, q[\"blocks\"], est[\"blocks\"])\n        b = b.astype({\"number\": \"int64\", \"ts\": \"int64\"}).sort_values(\"number\").reset_index(drop=True)\n        b[\"base_fee_per_gas\"] = pd.to_numeric(b[\"base_fee_per_gas\"], errors=\"coerce\").astype(\"float64\")\n        b.to_parquet(out / \"blocks.parquet\")\n        log(f\"blocks: {len(b):,}\")\n    for a, _ in MONTHS:\n        f = out / \"events\" / f\"{a[:7]}.parquet\"\n        if f.exists():\n            continue\n        t = time.time()\n        raw = _run(client, q[f\"logs {a[:7]}\"], est[f\"logs {a[:7]}\"])\n        ev = decode(raw)\n        del raw\n        ev.to_parquet(f)\n        log(f\"{a[:7]}: {len(ev):,} events ({time.time() - t:.0f}s)\")\n\n\ndef check_tokens(out):\n    \"\"\"Symbols and decimals in BigQuery's token table against the list above; mismatches are printed.\"\"\"\n    t = pd.read_csv(Path(out) / \"tokens.csv\", dtype=str)\n    by = {r.address.lower(): r for r in t.itertuples()}\n    bad = []\n    for s, (a, d, _) in TOKENS.items():\n        r = by.get(a)\n        if r is None:\n            bad.append(f\"{s}: address not in the token table\")\n        elif str(r.decimals).strip() not in (str(d), \"nan\", \"None\", \"\"):\n            bad.append(f\"{s}: decimals {r.decimals} in BigQuery, {d} here\")\n    return bad\n\n\ndef load_events(out):\n    ev = pd.concat([pd.read_parquet(f) for f in sorted((Path(out) / \"events\").glob(\"*.parquet\"))], ignore_index=True)\n    for k in (\"pool\", \"sender\"):\n        ev[k] = ev[k].astype(str).astype(\"category\")\n    return ev\n\n\ndef swap_counts(out):\n    \"\"\"Swaps per pool and week (pools x 26), cached in swaps_per_week.csv.\"\"\"\n    f = Path(out) / \"swaps_per_week.csv\"\n    if f.exists():\n        return pd.read_csv(f, index_col=0).rename(columns=int)\n    blocks = pd.read_parquet(Path(out) / \"blocks.parquet\", columns=[\"number\", \"ts\"])\n    wk = (blocks.ts.to_numpy() - T0) // WEEK\n    spw = []\n    for g in sorted((Path(out) / \"events\").glob(\"*.parquet\")):\n        e = pd.read_parquet(g, columns=[\"block_number\", \"pool\", \"kind\"])\n        e = e[e.kind == 0]\n        spw.append(pd.crosstab(e.pool.astype(str), wk[e.block_number.to_numpy() - blocks.number.iloc[0]]))\n    spw = pd.concat(spw).groupby(level=0).sum().reindex(columns=range(WEEKS), fill_value=0)\n    spw.to_csv(f)\n    return spw\n\n\n# ----------------------------------------------------------------------------------------------------------------------\n# Binance one-second klines\n\ndef _download(url, tries=4):\n    for k in range(tries):\n        try:\n            return urllib.request.urlopen(urllib.request.Request(url, headers={\"User-Agent\": \"p1a-dev\"}), timeout=300).read()\n        except urllib.error.HTTPError as e:\n            if e.code == 404:\n                return None\n            if k == tries - 1:\n                raise\n        except Exception:\n            if k == tries - 1:\n                raise\n        time.sleep(5 * (k + 1))\n\n\ndef parse_klines(raw):\n    \"\"\"(seconds since T0, close, quote volume, trade count) from one monthly zip; ms or us timestamps.\"\"\"\n    with zipfile.ZipFile(io.BytesIO(raw)) as z:\n        k = pd.read_csv(z.open(z.namelist()[0]), header=None, usecols=[0, 4, 7, 8])\n    k = k.apply(pd.to_numeric, errors=\"coerce\").dropna()           # drops a header row if a file has one\n    t = k[0].to_numpy(np.int64)\n    t = t // 1_000_000 if t.max() > 10 ** 14 else t // 1_000\n    return t - T0, k[4].to_numpy(np.float64), k[7].to_numpy(np.float64), k[8].to_numpy(np.int64)\n\n\ndef needed_symbols(pools_df, swaps_per_week):\n    \"\"\"Binance symbols of the pools with at least MIN_SWAPS_WEEK swaps in every week.\"\"\"\n    ok = swaps_per_week.index[(swaps_per_week >= MIN_SWAPS_WEEK).all(axis=1)]\n    p = pools_df[pools_df.pool.isin(ok)]\n    return sorted({s for s in pd.concat([p.bin0, p.bin1]).dropna()})\n\n\ndef binance(symbols, out, log=print):\n    \"\"\"Dense per-second closes (forward-filled) and weekly activity, one file per symbol.\"\"\"\n    d = Path(out) / \"binance\"\n    d.mkdir(parents=True, exist_ok=True)\n    for s in symbols:\n        f = d / f\"{s}.npz\"\n        if f.exists():\n            continue\n        t = time.time()\n        close = np.full(NSEC, np.nan, np.float32)\n        count = np.zeros(NSEC, np.int32)\n        qv = np.zeros(NSEC, np.float32)\n        missing = []\n        for a, b in MONTHS:\n            in_dev(a, b)\n            raw = _download(BINANCE_URL.format(s=s, m=a[:7]))\n            parts = [raw] if raw is not None else []\n            if raw is None:                       # monthly file missing: try the daily files of that month\n                for day in pd.date_range(a, b, freq=\"D\", inclusive=\"left\"):\n                    r = _download(BINANCE_DAILY_URL.format(s=s, d=day.strftime(\"%Y-%m-%d\")))\n                    if r is not None:\n                        parts.append(r)\n            if not parts:\n                missing.append(a[:7])\n                continue\n            for raw in parts:\n                i, c, v, n = parse_klines(raw)\n                keep = (i >= 0) & (i < NSEC)\n                close[i[keep]], qv[i[keep]], count[i[keep]] = c[keep], v[keep], n[keep]\n        idx = np.where(np.isnan(close), 0, np.arange(NSEC))\n        np.maximum.accumulate(idx, out=idx)\n        close = close[idx]\n        wk = np.arange(NSEC) // WEEK\n        active = np.bincount(wk, weights=(count > 0).astype(float), minlength=WEEKS) / np.bincount(wk, minlength=WEEKS)\n        qv_w = np.bincount(wk, weights=qv, minlength=WEEKS)\n        np.savez_compressed(f, close=close, active=active, qv=qv_w, missing=np.array(missing))\n        log(f\"{s}: {'missing ' + ','.join(missing) + '; ' if missing else ''}{time.time() - t:.0f}s\")\n\n\ndef load_binance(out, s):\n    if s is None or (isinstance(s, float) and math.isnan(s)):     # USDT itself\n        return {\"close\": np.ones(NSEC, np.float32), \"active\": np.ones(WEEKS), \"qv\": np.full(WEEKS, np.inf)}\n    z = np.load(Path(out) / \"binance\" / f\"{s}.npz\")\n    return {k: z[k] for k in (\"close\", \"active\", \"qv\")}\n\n\n# ----------------------------------------------------------------------------------------------------------------------\n# Pool-week measures\n\nLN2_96 = 96 * math.log(2.0)\nDEPTH_FACTOR = math.sqrt(1 + DEPTH_BAND) - 1 / math.sqrt(1 + DEPTH_BAND)\n\n\ndef pool_state(ev, nblocks, bn0):\n    \"\"\"Replays one pool's events in order. In-range liquidity is set by each swap and moved by mints and burns whose\n    range contains the current tick. Returns per-event and per-block (state after the block's last event) arrays.\"\"\"\n    kind = ev.kind.to_numpy().tolist()\n    sp, L, tick = ev.sqrtp.to_numpy().tolist(), ev.liq.to_numpy().tolist(), ev.tick.to_numpy().tolist()\n    tl, tu, amt = ev.tick_lower.to_numpy().tolist(), ev.tick_upper.to_numpy().tolist(), ev.amount.to_numpy().tolist()\n    n = len(kind)\n    pre_sp, post_sp, post_L = np.full(n, np.nan), np.full(n, np.nan), np.full(n, np.nan)\n    cs, cl, ct = math.nan, math.nan, None\n    for j in range(n):\n        k = kind[j]\n        if k == 0:\n            pre_sp[j] = cs\n            cs, cl, ct = sp[j], L[j], tick[j]\n        elif ct is not None and tl[j] <= ct < tu[j]:\n            cl = cl + amt[j] if k == 1 else max(cl - amt[j], 0.0)\n        post_sp[j], post_L[j] = cs, cl\n    bi = ev.block_number.to_numpy() - bn0\n    last = np.searchsorted(bi, np.arange(nblocks), side=\"right\") - 1       # last event at or before each block\n    ok = last >= 0\n    blk_sp, blk_L = np.full(nblocks, np.nan), np.full(nblocks, np.nan)\n    blk_sp[ok], blk_L[ok] = post_sp[last[ok]], post_L[last[ok]]\n    return pre_sp, post_sp, post_L, blk_sp, blk_L\n\n\ndef _slopes(x, y, g, ngroups):\n    \"\"\"OLS slope of y on x within each group, from sums.\"\"\"\n    n = np.bincount(g, minlength=ngroups).astype(float)\n    sx, sy = np.bincount(g, x, ngroups), np.bincount(g, y, ngroups)\n    sxx, sxy = np.bincount(g, x * x, ngroups), np.bincount(g, x * y, ngroups)\n    with np.errstate(invalid=\"ignore\", divide=\"ignore\"):\n        vx = sxx - sx * sx / n\n        return np.where((n > 2) & (vx > 0), (sxy - sx * sy / n) / vx, np.nan), n\n\n\ndef _winsorize_by_group(x, g, ngroups):\n    x = x.copy()\n    for k in range(ngroups):\n        m = g == k\n        if m.sum() > 10:\n            lo, hi = np.percentile(x[m], WINSOR)\n            x[m] = np.clip(x[m], lo, hi)\n    return x\n\n\ndef pool_weeks(ev, meta, blocks, px0, px1):\n    \"\"\"One pool: 26 rows of measures, and the week x sender table of arbitrage volume in USD.\"\"\"\n    bn, ts = blocks[\"number\"].to_numpy(), blocks[\"ts\"].to_numpy()\n    nb = len(bn)\n    pre_sp, post_sp, post_L, blk_sp, blk_L = pool_state(ev, nb, bn[0])\n    dln = (meta.dec0 - meta.dec1) * math.log(10.0)\n    sec = np.clip(ts - 1 - T0, 0, NSEC - 1)                                 # the second just before each block\n    c0, c1 = px0[\"close\"][sec].astype(float), px1[\"close\"][sec].astype(float)\n    ref = np.log(c0) - np.log(c1)\n    usd1 = c1                                                               # USD (USDT) price of token1\n    wk = (ts - T0) // WEEK\n    # per block: pool log price after the block, basis in bps, depth in USD\n    logp = 2 * (np.log(blk_sp) - LN2_96) + dln\n    x = (logp - ref) * 1e4\n    depth = blk_L * (blk_sp / 2 ** 96) * DEPTH_FACTOR / 10 ** meta.dec1 * usd1\n    valid = np.isfinite(x) & np.isfinite(depth)\n    # per swap: flow in bps of log price, the swap's move, the basis before it, USD volume\n    sw = ev.kind.to_numpy() == 0\n    bi = (ev.block_number.to_numpy() - bn[0])[sw]\n    a1 = ev.a1.to_numpy()[sw]\n    psp, qsp, pl = post_sp[sw], pre_sp[sw], post_L[sw]\n    with np.errstate(divide=\"ignore\", invalid=\"ignore\"):\n        f = np.where(pl > 0, 2 * a1 / (pl * psp / 2 ** 96) * 1e4, 0.0)\n        move = 2 * (np.log(psp) - np.log(qsp))\n        xpre = (2 * (np.log(qsp) - LN2_96) + dln - ref[bi]) * 1e4\n    vol = np.abs(a1) / 10 ** meta.dec1 * usd1[bi]\n    arb = np.isfinite(move) & np.isfinite(xpre) & (move * xpre < 0)\n    flow = np.bincount(bi, weights=f, minlength=nb)\n    # aggregate response: flow over blocks b+1..b+3 on the basis at b, separately in odd and even UTC hours;\n    # a basis observation whose window reaches into the next hour is dropped, so the halves share no flows\n    hour = ts // 3600\n    b = np.arange(nb - 3)\n    same = (hour[b] == hour[b + 3]) & valid[b]\n    y = flow[b + 1] + flow[b + 2] + flow[b + 3]\n    rows = []\n    g = np.where(same, wk[b] * 2 + (hour[b] % 2), -1)\n    m = g >= 0\n    xw = _winsorize_by_group(x[b][m], g[m], 2 * WEEKS)\n    s_half, n_half = _slopes(xw, y[m], g[m], 2 * WEEKS)\n    gl = np.where(same, wk[b], -1)\n    ml = gl >= 0\n    xl = _winsorize_by_group(x[b][ml], gl[ml], WEEKS)\n    lag = [_slopes(xl, flow[b + k][ml], gl[ml], WEEKS)[0] for k in (1, 2, 3)]\n    # depth: time-weighted mean over blocks, the state taken after each block's last event, which leaves out\n    # liquidity minted and burned within one block (JIT)\n    dt = np.diff(ts, append=ts[-1] + 12).astype(float)\n    wv = np.where(valid, dt, 0.0)\n    dep = np.bincount(wk, wv * np.nan_to_num(depth), WEEKS) / np.bincount(wk, wv, WEEKS)\n    # Binance controls\n    r = np.diff(np.log(px0[\"close\"].astype(float)) - np.log(px1[\"close\"].astype(float)), prepend=np.nan)\n    r = np.nan_to_num(r)\n    rv = np.bincount(np.arange(NSEC) // WEEK, r * r, WEEKS)\n    legs = [p[\"qv\"] for p in (px0, px1) if np.isfinite(p[\"qv\"]).all()]\n    qv = np.min(legs, axis=0) if legs else np.full(WEEKS, np.nan)\n    swk = wk[bi]\n    nsw = np.bincount(swk, minlength=WEEKS)\n    vusd = np.bincount(swk, vol, WEEKS)\n    for w in range(WEEKS):\n        rows.append({\"pool\": meta.pool, \"pair\": meta.pair, \"fee\": meta.fee, \"week\": w,\n                     \"R1\": -s_half[2 * w + 1], \"R2\": -s_half[2 * w], \"n1\": n_half[2 * w + 1], \"n2\": n_half[2 * w],\n                     \"R_lag1\": -lag[0][w], \"R_lag2\": -lag[1][w], \"R_lag3\": -lag[2][w],\n                     \"D\": math.log(dep[w]) if dep[w] > 0 else np.nan, \"swaps\": int(nsw[w]), \"volume_usd\": vusd[w],\n                     \"log_rv\": math.log(rv[w]) if rv[w] > 0 else np.nan,\n                     \"log_cex_volume\": math.log(qv[w]) if qv[w] > 0 else np.nan,\n                     \"active0\": px0[\"active\"][w], \"active1\": px1[\"active\"][w]})\n    pw = pd.DataFrame(rows)\n    pw.attrs[\"median_abs_basis_bps\"] = float(np.nanmedian(np.abs(x[valid]))) if valid.any() else np.nan\n    pw.attrs[\"median_basis_bps\"] = float(np.nanmedian(x[valid])) if valid.any() else np.nan\n    sender = ev.sender.to_numpy()[sw]\n    arbv = pd.DataFrame({\"week\": swk[arb], \"sender\": sender[arb], \"usd\": vol[arb]})\n    arbv = arbv.groupby([\"week\", \"sender\"], observed=True).usd.sum().reset_index()\n    arbv[\"pool\"] = meta.pool\n    return pw, arbv\n\n\ndef composition(arbv):\n    \"\"\"Per pool-week: Herfindahl index of arbitrage volume across senders, top share, senders with >= 1%, top change.\"\"\"\n    rows = []\n    for (p, w), g in arbv.groupby([\"pool\", \"week\"]):\n        s = g.usd.to_numpy() / g.usd.sum()\n        o = np.argsort(-s)\n        rows.append({\"pool\": p, \"week\": w, \"C\": float((s * s).sum()), \"top_share\": float(s[o[0]]),\n                     \"n_1pct\": int((s >= 0.01).sum()), \"top_sender\": g.sender.to_numpy()[o[0]]})\n    c = pd.DataFrame(rows).sort_values([\"pool\", \"week\"])\n    c[\"top_changed\"] = (c.groupby(\"pool\").top_sender.shift() != c.top_sender) & c.groupby(\"pool\").week.diff().eq(1)\n    return c\n\n\ndef instrument(arbv, pools):\n    \"\"\"Leave-one-out shift-share Z on weeks 14-26: exposures are each bot's share of the pool's arbitrage volume in\n    weeks 1-13; shifts are the bot's standardized log arbitrage volume in all other eligible pools.\"\"\"\n    a = arbv[arbv.pool.isin(pools)]\n    pre, post = a[a.week < EXPOSURE_WEEKS], a[a.week >= EXPOSURE_WEEKS]\n    bots = pre.groupby(\"sender\", observed=True).usd.sum().sort_values(ascending=False).index[:N_BOTS]\n    expo = pre[pre.sender.isin(bots)].groupby([\"pool\", \"sender\"], observed=True).usd.sum()\n    expo = (expo / pre.groupby(\"pool\").usd.sum().reindex(expo.index.get_level_values(0)).to_numpy()).unstack(fill_value=0.0)\n    weeks = np.arange(EXPOSURE_WEEKS, WEEKS)\n    tot = post[post.sender.isin(bots)].pivot_table(index=\"sender\", columns=\"week\", values=\"usd\", aggfunc=\"sum\",\n                                                   fill_value=0.0, observed=True).reindex(index=bots, columns=weeks, fill_value=0.0)\n    own = post[post.sender.isin(bots)].groupby([\"pool\", \"sender\", \"week\"], observed=True).usd.sum()\n    rows = []\n    for p in pools:\n        z = np.zeros(len(weeks))\n        if p in expo.index:\n            for bname, s in expo.loc[p].items():\n                if s <= 0:\n                    continue\n                v = tot.loc[bname].to_numpy() - np.array([own.get((p, bname, w), 0.0) for w in weeks])\n                A = np.log1p(np.maximum(v, 0.0))\n                sd = A.std()\n                z += s * ((A - A.mean()) / sd if sd > 0 else 0.0)\n        rows += [{\"pool\": p, \"week\": int(w), \"Z\": float(zz)} for w, zz in zip(weeks, z)]\n    return pd.DataFrame(rows)\n\n\ndef build_panel(out, pools_df, log=print):\n    \"\"\"All pool-weeks of the pools that pass the swap-count floor, with the floors' inputs; saved to panel.parquet.\"\"\"\n    out = Path(out)\n    f = out / \"panel.parquet\"\n    if f.exists():\n        return pd.read_parquet(f), pd.read_parquet(out / \"arb_volume.parquet\")\n    blocks = pd.read_parquet(out / \"blocks.parquet\")\n    assert np.all(np.diff(blocks.number.to_numpy()) == 1), \"block numbers are not contiguous\"\n    spw = swap_counts(out)\n    keep = spw.index[(spw >= MIN_SWAPS_WEEK).all(axis=1)]\n    ev = load_events(out)\n    ev = ev[ev.pool.isin(keep)].sort_values([\"block_number\", \"log_index\"]).reset_index(drop=True)\n    log(f\"pools with events: {spw.shape[0]}; with >= {MIN_SWAPS_WEEK} swaps in every week: {len(keep)}\")\n    cache, panels, arbs = {}, [], []\n    for meta in pools_df[pools_df.pool.isin(keep)].itertuples():\n        for s in (meta.bin0, meta.bin1):\n            if s not in cache:\n                cache[s] = load_binance(out, s)\n        t = time.time()\n        pw, arbv = pool_weeks(ev[ev.pool == meta.pool], meta, blocks, cache[meta.bin0], cache[meta.bin1])\n        pw[\"median_basis_bps\"] = pw.attrs[\"median_basis_bps\"]\n        panels.append(pw)\n        arbs.append(arbv)\n        log(f\"{meta.pair} {meta.fee}: median basis {pw.attrs['median_basis_bps']:+.1f} bps ({time.time() - t:.0f}s)\")\n    panel = pd.concat(panels, ignore_index=True)\n    arbv = pd.concat(arbs, ignore_index=True)\n    panel = panel.merge(composition(arbv), on=[\"pool\", \"week\"], how=\"left\")\n    panel.to_parquet(f)\n    arbv.to_parquet(out / \"arb_volume.parquet\")\n    return panel, arbv\n\n\n# ----------------------------------------------------------------------------------------------------------------------\n# Calibration\n\ndef demean2(df, col):\n    \"\"\"Two-way within transformation on a balanced pool x week panel.\"\"\"\n    m = df.pivot(index=\"pool\", columns=\"week\", values=col).to_numpy(float)\n    return (m - m.mean(1, keepdims=True) - m.mean(0, keepdims=True) + m.mean()).ravel()\n\n\ndef eligible(panel, volume_floor):\n    g = panel.groupby(\"pool\")\n    ok = (g.swaps.min() >= MIN_SWAPS_WEEK) & (g.median_basis_bps.first().abs() <= MAX_MEDIAN_BASIS_BPS) \\\n        & (g.active0.mean() >= MIN_ACTIVE_SHARE) & (g.active1.mean() >= MIN_ACTIVE_SHARE) \\\n        & (g.volume_usd.median() >= volume_floor)\n    full = g.apply(lambda d: d[[\"R1\", \"R2\", \"D\", \"C\"]].notna().all().all() and len(d) == WEEKS)\n    return ok.index[ok & full]\n\n\ndef _cluster_t(y, X, cl):\n    \"\"\"OLS with cluster-robust (CR1) standard errors; returns coefficients and t statistics.\"\"\"\n    b = np.linalg.lstsq(X, y, rcond=None)[0]\n    u = y - X @ b\n    A = np.linalg.inv(X.T @ X)\n    G = np.unique(cl)\n    meat = sum(np.outer(X[cl == c].T @ u[cl == c], X[cl == c].T @ u[cl == c]) for c in G)\n    n, k = X.shape\n    V = len(G) / (len(G) - 1) * (n - 1) / (n - k) * A @ meat @ A\n    return b, b / np.sqrt(np.diag(V))\n\n\ndef calibrate(out, panel, arbv, log=print):\n    out = Path(out)\n    res = {\"window\": [DEV_START, DEV_END], \"floors\": {\"min_swaps_week\": MIN_SWAPS_WEEK, \"min_active_share\": MIN_ACTIVE_SHARE,\n                                                      \"max_median_basis_bps\": MAX_MEDIAN_BASIS_BPS, \"winsor\": WINSOR},\n           \"counts\": {}, \"note\": \"No estimate of composition's effect on depth is computed here, on purpose.\",\n           \"module_sha256\": hashlib.sha256(Path(__file__).read_bytes()).hexdigest()}\n    for vf in VOLUME_FLOORS_USD:\n        e = eligible(panel, vf)\n        res[\"counts\"][f\"volume_floor_{int(vf):d}\"] = {\"pools\": int(len(e)),\n                                                      \"pairs\": int(panel[panel.pool.isin(e)].pair.nunique())}\n    vf = VOLUME_FLOORS_USD[0]\n    e = list(eligible(panel, vf))\n    res[\"volume_floor_used\"] = vf\n    if len(e) < 3:\n        res[\"error\"] = \"fewer than 3 eligible pools; nothing to calibrate\"\n        (out / \"calibration.json\").write_text(json.dumps(res, indent=1))\n        return res\n    p = panel[panel.pool.isin(e)].sort_values([\"pool\", \"week\"]).reset_index(drop=True)\n    p[\"Rbar\"] = (p.R1 + p.R2) / 2\n    r1, r2, rb, c, d = (demean2(p, k) for k in (\"R1\", \"R2\", \"Rbar\", \"C\", \"D\"))\n    rho_half = float(np.corrcoef(r1, r2)[0, 1])\n    rel_avg = 2 * rho_half / (1 + rho_half) if rho_half > 0 else float(\"nan\")\n    psi = float(np.corrcoef(c, rb)[0, 1])\n    bR = float(np.dot(rb, d) / np.dot(rb, rb))\n    eps = (d - bR * rb).reshape(len(e), WEEKS)\n    phi = float(np.corrcoef(eps[:, 1:].ravel(), eps[:, :-1].ravel())[0, 1])\n    z = instrument(arbv, e)\n    q = p[p.week >= EXPOSURE_WEEKS].merge(z, on=[\"pool\", \"week\"], how=\"left\")\n    cz, zz, r2z = demean2(q, \"C\"), demean2(q, \"Z\"), demean2(q, \"R2\")\n    pairs = q.pivot(index=\"pool\", columns=\"week\", values=\"pair\").to_numpy().ravel()\n    tz = _cluster_t(cz, np.column_stack([zz, r2z]), pairs)[1] if len(set(pairs)) > 1 else [float(\"nan\")]\n    res.update({\n        \"pools\": e, \"n_pools\": len(e), \"n_pairs\": int(p.pair.nunique()),\n        \"reliability\": {\"half_correlation_within\": rho_half, \"average_within_spearman_brown\": rel_avg},\n        \"composition\": {\"sd_within\": float(c.std()), \"corr_with_response_within\": psi,\n                        \"corr_with_true_response_disattenuated\": psi / math.sqrt(rel_avg) if rel_avg > 0 else None},\n        \"response\": {\"sd_within\": float(rb.std()), \"depth_slope_per_unit\": bR, \"depth_effect_per_sd\": bR * float(rb.std())},\n        \"depth\": {\"residual_sd_within\": float(eps.std()), \"residual_ar1_within_pool\": phi},\n        \"instrument\": {\"weeks\": [EXPOSURE_WEEKS + 1, WEEKS], \"corr_with_composition_within\": float(np.corrcoef(cz, zz)[0, 1]),\n                       \"first_stage_t_clustered_by_pair\": float(tz[0]), \"first_stage_F\": float(tz[0] ** 2)},\n    })\n    (out / \"calibration.json\").write_text(json.dumps(res, indent=1))\n    return res\n\n\ndef summary_for_chat(res):\n    \"\"\"The aggregate numbers only, to paste back into the conversation.\"\"\"\n    keep = (\"window\", \"module_sha256\", \"volume_floor_used\", \"counts\", \"n_pools\", \"n_pairs\", \"reliability\", \"composition\", \"response\",\n            \"depth\", \"instrument\", \"error\")\n    return json.dumps({k: res[k] for k in keep if k in res}, indent=1)\n"
assert hashlib.sha256(P1A_DEV_SRC.encode()).hexdigest() == P1A_DEV_SHA256, "the pipeline code changed"
open("/content/p1a_dev.py", "w").write(P1A_DEV_SRC)
sys.path.insert(0, "/content")
import p1a_dev as M
importlib.reload(M)
from google.cloud import bigquery
assert PROJECT_ID, "set PROJECT_ID in step 1"
client = bigquery.Client(project=PROJECT_ID)
os.makedirs(OUT_DIR, exist_ok=True)
print("ready; pipeline", P1A_DEV_SHA256[:12])

In [ ]:
# Step 3. Candidate pools, and dry runs (free): how much each query would scan
pools = M.candidate_pools()
print(len(pools), "candidate pools in", pools.pair.nunique(), "token pairs; the pool-address self-test passed")
est_file = os.path.join(OUT_DIR, "estimates.json")
if os.path.exists(est_file):
    est = json.load(open(est_file))
else:
    est = M.estimate(client, pools.pool.tolist())
    json.dump(est, open(est_file, "w"))
for name, b in est:
    print(f"  {name:12s} {b / 1e9:9.1f} GB")
total = sum(b for _, b in est)
print(f"total {total / 1e9:.1f} GB; this notebook's budget {M.BUDGET_BYTES / 1e9:.0f} GB")
print("OK to run step 4 once I_ACCEPT_THE_SCAN = True" if total <= M.BUDGET_BYTES else
      "ABOVE THE BUDGET: stop here and send these numbers to Claude")

In [ ]:
# Step 4. The queries (development window only), month by month, each capped at its estimate
assert total <= M.BUDGET_BYTES, "the scan is above the budget: stop here and send the numbers from step 3 to Claude"
assert I_ACCEPT_THE_SCAN, "read the estimate in step 3, then set I_ACCEPT_THE_SCAN = True in step 1 and rerun"
M.extract(client, OUT_DIR, pools.pool.tolist(), est)
bad = M.check_tokens(OUT_DIR)
print("token check:", bad if bad else "every symbol and decimals value matches BigQuery's token table")

In [ ]:
# Step 5. Binance one-second prices for the pools that trade in every week (public archive, no account)
import urllib.request
try:
    urllib.request.urlopen("https://data.binance.vision/data/spot/monthly/klines/ETHUSDT/1s/ETHUSDT-1s-2024-01.zip.CHECKSUM",
                           timeout=60).read()
except Exception as e:
    raise SystemExit(f"data.binance.vision is not reachable from this session ({e}); send this message to Claude")
spw = M.swap_counts(OUT_DIR)
syms = M.needed_symbols(pools, spw)
print(len(spw), "pools have swaps;", int((spw >= M.MIN_SWAPS_WEEK).all(axis=1).sum()),
      f"have at least {M.MIN_SWAPS_WEEK} in every week; Binance symbols needed:", len(syms))
M.binance(syms, OUT_DIR)

In [ ]:
# Step 6. Pool-week measures: response halves, composition, depth, controls
panel, arbv = M.build_panel(OUT_DIR, pools)
print(panel.pool.nunique(), "pools,", len(panel), "pool-weeks")

In [ ]:
# Step 7. Calibration. Copy everything printed below into the chat with Claude.
res = M.calibrate(OUT_DIR, panel, arbv)
print(M.summary_for_chat(res))